# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [34]:
data = pd.read_csv('./data/AviationData.csv', index_col=0, encoding='latin1')
data.info()
data.head()
data.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 88889 entries, 20001218X45444 to 20221230106513
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Investigation.Type      88889 non-null  object 
 1   Accident.Number         88889 non-null  object 
 2   Event.Date              88889 non-null  object 
 3   Location                88837 non-null  object 
 4   Country                 88663 non-null  object 
 5   Latitude                34382 non-null  object 
 6   Longitude               34373 non-null  object 
 7   Airport.Code            50132 non-null  object 
 8   Airport.Name            52704 non-null  object 
 9   Injury.Severity         87889 non-null  object 
 10  Aircraft.damage         85695 non-null  object 
 11  Aircraft.Category       32287 non-null  object 
 12  Registration.Number     87507 non-null  object 
 13  Make                    88826 non-null  object 
 14  Model                

C:\Users\tying\AppData\Local\Temp\ipykernel_24604\4093557294.py:1: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('./data/AviationData.csv', index_col=0, encoding='latin1')


,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [35]:
recent_data = data[data['Event.Date'] >= '1983-01-01']
print(recent_data['Event.Date'].head())
print(recent_data['Amateur.Built'].head())
pro_recent_data = recent_data[recent_data['Amateur.Built'] == 'No']
pro_recent_data['Amateur.Built'].value_counts()

pro_recent_data = pro_recent_data[pro_recent_data['Aircraft.Category'] == 'Airplane']
pro_recent_data['Aircraft.Category'].value_counts()
#print(pro_recent_data.columns)

Event.Id
20001214X42040    1983-01-01
20001214X42095    1983-01-01
20001214X42067    1983-01-01
20001214X42063    1983-01-01
20001214X42018    1983-01-01
Name: Event.Date, dtype: object
Event.Id
20001214X42040    No
20001214X42095    No
20001214X42067    No
20001214X42063    No
20001214X42018    No
Name: Amateur.Built, dtype: object


Aircraft.Category
Airplane    21447
Name: count, dtype: int64

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

### Assumptions

The assumption is that the total number of passengers on a flight is equal to the sum of the total fatal, total serious, total minor and total uninjured columns. Because of this, we can find the percent of serious and fatal injuires by dividing by the total.

In [36]:
total_passenger = pro_recent_data['Total.Fatal.Injuries'] + pro_recent_data['Total.Serious.Injuries'] + pro_recent_data['Total.Minor.Injuries'] + pro_recent_data['Total.Uninjured']
pro_recent_data['Total.Passengers'] = total_passenger

# Calculate the percentage of serious injuries and fatal injuries out of the total passengers
percent_serious_injury = pro_recent_data['Total.Serious.Injuries'] / total_passenger
percent_fatal_injury = pro_recent_data['Total.Fatal.Injuries'] / total_passenger
pro_recent_data['Percent.Serious.Or.Fatal.Injury'] = percent_serious_injury + percent_fatal_injury
#pro_recent_data['Percent.Fatal.Injury'] = percent_fatal_injury
pro_recent_data.head()

,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,...,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.Passengers,Percent.Serious.Or.Fatal.Injury
Event.Id,,,,,,,,,,,,,,,,,,,,,
20001214X42478,Incident,LAX83IA149B,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,Incident,...,NaN,NaN,NaN,588.0,VMC,Standing,Probable Cause,04-12-2014,NaN,NaN
20001214X42478,Incident,LAX83IA149A,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,Incident,...,NaN,NaN,NaN,588.0,VMC,Taxi,Probable Cause,04-12-2014,NaN,NaN
20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,Fatal(1),...,1.0,1.0,NaN,NaN,IMC,Cruise,Probable Cause,02-05-2011,NaN,NaN
20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,NaN,NaN,TX05,AERO COUNTRY,Fatal(1),...,1.0,NaN,NaN,4.0,VMC,Standing,Probable Cause,17-10-2016,NaN,NaN
20001214X44248,Incident,MIA83IA210,1983-08-21,"NORFOLK, VA",United States,NaN,NaN,NaN,NaN,Incident,...,NaN,NaN,NaN,289.0,VMC,Cruise,Probable Cause,01-02-2016,NaN,NaN


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [37]:
#We can assume unknown does not mean destroyed because it is less than 100 out of ~65000
print(pro_recent_data['Aircraft.damage'].value_counts())
#Destoryed tells us if the plane is destroyed or not, so we can use it to create a new column called 'Destroyed' with boolean values.
pro_recent_data['Destroyed'] = pro_recent_data['Aircraft.damage'] == 'Destroyed'
pro_recent_data['Destroyed'].value_counts()

Aircraft.damage
Substantial    16990
Destroyed       2316
Minor            817
Unknown           97
Name: count, dtype: int64


Destroyed
False    19131
True      2316
Name: count, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [38]:
pro_recent_data['Make'] = pro_recent_data['Make'].str.title()
pro_recent_data = pro_recent_data[pro_recent_data.groupby('Make')['Make'].transform('count') >= 50]
pro_recent_data['Make'].value_counts()
pro_recent_data.replace('Bombardier Inc', 'Bombardier', inplace=True)

pro_recent_data.replace('Piper Aircraft, Inc.', 'Piper', inplace=True)
pro_recent_data.replace('Beechcraft Corporation', 'Beech', inplace=True)
pro_recent_data.replace('Air Tractor Inc', 'Air Tractor', inplace=True)
pro_recent_data.replace('Aviat Aircraft Inc', 'Aviat', inplace=True)
pro_recent_data.replace('De Havilland', 'Dehavilland', inplace=True)
pro_recent_data['Make'].value_counts()

Make
Cessna                            7146
Piper                             3989
Beech                             1431
Boeing                            1264
Air Tractor                        425
Mooney                             363
Airbus                             243
Cirrus Design Corp                 220
Bellanca                           219
Maule                              215
Aeronca                            200
Dehavilland                        168
Champion                           158
Embraer                            153
Grumman                            147
Aviat                              146
Luscombe                           141
Cirrus                             137
Stinson                            129
Bombardier                         115
Mcdonnell Douglas                  108
North American                     106
Taylorcraft                         93
Aero Commander                      90
Socata                              75
Diamond Aircraft Ind

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [39]:
print(pro_recent_data['Model'].value_counts())
pro_recent_data.dropna(subset=['Model'], inplace=True)
print((pro_recent_data.groupby('Model')['Make'].nunique() == 1).all())
make_model = pro_recent_data['Make'] + ' ' + pro_recent_data['Model']
pro_recent_data['Make.Model'] = make_model
pro_recent_data['Model'].isnull().sum()

Model
172          769
737          403
152          316
182          304
172S         276
            ... 
747-451        1
310-J          1
777 - 200      1
D-45           1
PA-44          1
Name: count, Length: 2034, dtype: int64
False


0

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [40]:
#print(pro_recent_data['Engine.Type'].value_counts())
pro_recent_data['Engine.Type'].replace(['UNK', 'LR'], 'Unknown', inplace=True)
#print(pro_recent_data['Engine.Type'].value_counts())

#print(pro_recent_data['Weather.Condition'].value_counts())
pro_recent_data['Weather.Condition'].replace('Unk', 'UNK', inplace=True)
#print(pro_recent_data['Weather.Condition'].value_counts())

print(pro_recent_data['Broad.phase.of.flight'].value_counts())
#Zero represents unknown, so we can change it to NaN
print(pro_recent_data['Number.of.Engines'].value_counts())
pro_recent_data['Number.of.Engines'].replace(0, np.nan, inplace=True)
print(pro_recent_data['Number.of.Engines'].value_counts())


Broad.phase.of.flight
Landing        1110
Takeoff         425
Cruise          238
Approach        210
Maneuvering     127
Taxi             99
Go-around        81
Descent          62
Climb            52
Standing         35
Unknown          11
Other             2
Name: count, dtype: int64
Number.of.Engines
1.0    13222
2.0     2470
4.0       67
3.0       26
0.0        5
Name: count, dtype: int64
Number.of.Engines
1.0    13222
2.0     2470
4.0       67
3.0       26
Name: count, dtype: int64


C:\Users\tying\AppData\Local\Temp\ipykernel_24604\1350926578.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  pro_recent_data['Engine.Type'].replace(['UNK', 'LR'], 'Unknown', inplace=True)
C:\Users\tying\AppData\Local\Temp\ipykernel_24604\1350926578.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alwa

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [41]:
print(pro_recent_data.isna().sum().loc[lambda x: x > 2000])
pro_recent_data.drop(columns=['Airport.Code','Airport.Name', 'Air.carrier', 'Schedule'], inplace=True)

Airport.Code                        6231
Airport.Name                        6125
Number.of.Engines                   2094
Engine.Type                         3214
Schedule                           15740
Purpose.of.flight                   3047
Air.carrier                         9431
Total.Fatal.Injuries                2386
Total.Serious.Injuries              2458
Total.Minor.Injuries                2223
Weather.Condition                   2417
Broad.phase.of.flight              15427
Report.Status                       3785
Total.Passengers                    2620
Percent.Serious.Or.Fatal.Injury     3360
dtype: int64


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [42]:
pro_recent_data.to_csv('./data/cleaned_aviation_data.csv')